# Test

## SIR 3S Toolkit

### Regular Import/Init

In [1]:
from sir3stoolkit.core import wrapper

In [2]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [3]:
wrapper.Initialize_Toolkit(r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2")

[2026-06-26 11:26:14,654] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-26 11:26:14,654] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-26 11:26:14,722] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


### Additional Import/Init for Dataframes class

In [4]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [5]:
s3s = SIR3S_Model_Dataframes()

[2026-06-26 11:26:30,359] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


## Additional

In [6]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx
import logging

# Open Model

In [7]:
from pathlib import Path

network_configs = [
    ("Water", s3s.NetworkType.Water),
    ("DistrictHeating", s3s.NetworkType.DistrictHeating),
    ("Gas", s3s.NetworkType.Gas),
    ("Steam", s3s.NetworkType.Steam),
]

output_dir = (Path.cwd().resolve() / "./output/.").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

network_configs

[('Water', <NetworkType.Water: 1>),
 ('DistrictHeating', <NetworkType.DistrictHeating: 21>),
 ('Gas', <NetworkType.Gas: 31>),
 ('Steam', <NetworkType.Steam: 35>)]

## 1) Collect Per Network, Build Global Comparison, Then Map
1) Collect object properties/result properties per network.
2) Build a global object-type comparison and conflict report (properties and value types).
3) Apply table-name mapping afterwards as an independent step.

In [8]:
import json
import re
from pathlib import Path

all_reports = []
network_object_data = {}

def normalize_type(value_type):
    if value_type is None:
        return "Unknown"
    text = str(value_type).strip()
    return text if text else "Unknown"

def filter_unknown_types(type_iterable):
    return sorted({t for t in type_iterable if t != "Unknown"})

def map_object_type_headings(rst_text, table_name_mapping):
    heading_pattern = re.compile(r"^(?P<name>[^\r\n]+)\r?\n(?P<uline>\^{3,})\r?\n", re.MULTILINE)
    transformed_parts = []
    last_idx = 0
    mapped_count = 0
    error_count = 0

    for match in heading_pattern.finditer(rst_text):
        start, end = match.span()
        old_name = match.group("name").strip()
        transformed_parts.append(rst_text[last_idx:start])

        if old_name in table_name_mapping:
            new_heading = table_name_mapping[old_name]
            mapped_count += 1
        else:
            new_heading = f"ERROR_NO_MAPPING__{old_name}"
            error_count += 1

        transformed_parts.append(new_heading + "\n")
        transformed_parts.append("^" * len(new_heading) + "\n")
        transformed_parts.append(f"Object Type: ``{old_name}``\n\n")
        last_idx = end

    transformed_parts.append(rst_text[last_idx:])
    mapped_text = "".join(transformed_parts)
    return mapped_text, mapped_count, error_count

# Build table-name mapping once and use it for all mapped output files.
table_name_mapping = {}
for i in range(0, 150):
    try:
        object_type = s3s.ObjectTypes(i)
        table_name = s3s.ObjectTypes_TableNames(i)
        table_name_mapping[object_type.name] = table_name.name
    except Exception:
        pass

# Phase 1: collect per-network object data and write per-network RST files.
for network_name, network_enum in network_configs:
    db_path = f".\\network_type_{network_name}.db3"
    print(f"=== Collecting network: {network_name} ===")

    network_object_data[network_name] = {}

    try:
        s3s.NewModel(
            dbName=db_path,
            providerType=s3s.ProviderTypes.SQLite,
            namedInstance="",
            netType=network_enum,
            modelDescription=f"Tutorial New Model - {network_name}",
            userID="",
            password="",
        )
        s3s.StartTransaction("Inserting Nodes")

        node10 = s3s.AddNewNode(tkCont="-1", name="Node10", typ="QKON", x=200, y=700, z=213, qm_PH=0, symbolFactor=1, description="DNODE 10", idRef="RefNode 10", kvr=0)
        node11 = s3s.AddNewNode(tkCont="-1", name="Node11", typ="QKON", x=300, y=700, z=213, qm_PH=0, symbolFactor=1, description="DNODE 11", idRef="RefNode 11", kvr=0)

        s3s.EndTransaction()
        s3s.SaveChanges()

        object_types = [
            item
            for item in dir(s3s.ObjectTypes)
            if not (item.startswith("__") and item.endswith("__"))
            and item not in {"name", "value"}
            and not item.startswith("_")
        ]

        # 1) Gather property/result names and insert all possible elements first.
        object_records = {}
        for ot in object_types:
            props = []
            results = []
            tk = None
            insert_error = None

            try:
                props = s3s.GetPropertiesofElementType(s3s.ObjectTypes[ot]) or []
            except Exception as ex:
                props = []
                insert_error = f"GetProperties error: {ex}"

            try:
                results = s3s.GetResultProperties_from_elementType(s3s.ObjectTypes[ot], False) or []
            except Exception as ex:
                results = []
                if insert_error is None:
                    insert_error = f"GetResultProperties error: {ex}"

            try:
                tk = s3s.InsertElement(s3s.ObjectTypes[ot], "-1")
            except Exception as ex:
                if insert_error is None:
                    insert_error = f"InsertElement error: {ex}"

            object_records[ot] = {
                "props": list(props),
                "results": list(results),
                "tk": tk,
                "insert_error": insert_error,
            }

        # 2) Render RST.
        rst_lines = []
        rst_lines.append("Object Types, Properties, and Result Value Types")
        rst_lines.append("-----------------------------------------------")
        rst_lines.append("")
        rst_lines.append(f".. note:: Network Type: {network_name}")
        rst_lines.append("   The below sections lists all table names from self.ObjectTypes_TableNames, along with their properties, result properties, and respective object type from self.ObjectTypes (needed for toolkit operations like self.InsertElement(), self.GetPropertiesOfElementType()). The value types of the properties are also listed.")
        rst_lines.append("   Every property both model data and result will be returned as a python str. For result properties, this report intentionally lists names only.")
        rst_lines.append("")

        for ot in object_types:
            record = object_records[ot]
            props = record["props"]
            results = record["results"]
            tk = record["tk"]
            insert_error = record["insert_error"]

            rst_lines.append(ot)
            rst_lines.append("^" * len(ot))
            rst_lines.append("")

            prop_map = {}
            result_map = {}

            rst_lines.append("Properties")
            rst_lines.append('"' * len("Properties"))
            rst_lines.append("")
            if props:
                rst_lines.append(".. list-table::")
                rst_lines.append("   :header-rows: 1")
                rst_lines.append("")
                rst_lines.append("   * - Name")
                rst_lines.append("     - Value Type")
                for prop in props:
                    value_type = "Unknown"
                    if tk is not None:
                        try:
                            value_type = s3s.GetValue(tk, prop)[1]
                        except Exception:
                            value_type = "Unknown"
                    value_type = normalize_type(value_type)
                    prop_map[prop] = value_type
                    rst_lines.append(f"   * - ``{prop}``")
                    rst_lines.append(f"     - ``{value_type}``")
            else:
                rst_lines.append("No properties found.")
            rst_lines.append("")

            rst_lines.append("Result Properties")
            rst_lines.append('"' * len("Result Properties"))
            rst_lines.append("")
            if results:
                rst_lines.append(".. list-table::")
                rst_lines.append("   :header-rows: 1")
                rst_lines.append("")
                rst_lines.append("   * - Name")
                for result_name in results:
                    result_map[result_name] = "Unknown"
                    rst_lines.append(f"   * - ``{result_name}``")
            else:
                rst_lines.append("No result properties found.")
            rst_lines.append("")

            if insert_error:
                rst_lines.append(f".. warning:: {insert_error}")
                rst_lines.append("")

            network_object_data[network_name][ot] = {
                "properties": prop_map,
                "result_properties": result_map,
            }

        # 3) Delete all temporary elements after reading.
        for rec in object_records.values():
            tk = rec["tk"]
            if tk is not None:
                try:
                    s3s.DeleteElement(tk)
                except Exception:
                    pass

        network_snippet_text = "\n".join(rst_lines) + "\n"
        network_output_path = output_dir / f"object_types_props_results_snippet_{network_name}.rst"
        network_output_path.write_text(network_snippet_text, encoding="utf-8")

        network_mapped_text, network_mapped_count, network_error_count = map_object_type_headings(
            network_snippet_text,
            table_name_mapping,
        )
        network_mapped_output_path = output_dir / f"object_types_props_results_snippet_{network_name}_mapped.rst"
        network_mapped_output_path.write_text(network_mapped_text, encoding="utf-8")

        all_reports.append({
            "network": network_name,
            "network_output": str(network_output_path),
            "network_mapped_output": str(network_mapped_output_path),
            "object_types": len(network_object_data[network_name]),
            "mapped_headings": network_mapped_count,
            "mapping_errors": network_error_count,
        })

    finally:
        try:
            s3s.CloseModel(False)
        except Exception:
            pass

# Phase 2: global comparison across networks (independent of table-name mapping).
all_network_names = [name for name, _ in network_configs]
all_object_types = sorted({
    ot
    for data in network_object_data.values()
    for ot in data.keys()
})

global_rst_lines = []
global_rst_lines.append("Object Types, Properties, and Result Value Types")
global_rst_lines.append("---------------------------------------------")
global_rst_lines.append("")
global_rst_lines.append(".. note:: Aggregated global view across configured network types.")
global_rst_lines.append("   The below sections lists all table names from self.ObjectTypes_TableNames, along with their properties, result properties, and respective object type from self.ObjectTypes (needed for toolkit operations like self.InsertElement(), self.GetPropertiesOfElementType()).")
global_rst_lines.append("   Result properties are listed by name only.")
global_rst_lines.append("")

property_presence_conflicts = []
property_type_conflicts = []
result_presence_conflicts = []
result_type_conflicts = []

for ot in all_object_types:
    global_rst_lines.append(ot)
    global_rst_lines.append("^" * len(ot))
    global_rst_lines.append("")

    prop_entries = {}
    result_entries = {}

    for network_name in all_network_names:
        ot_data = network_object_data.get(network_name, {}).get(ot, {})

        for prop_name, prop_type in ot_data.get("properties", {}).items():
            if prop_name not in prop_entries:
                prop_entries[prop_name] = {"networks": set(), "types": set(), "typed_networks": set()}
            prop_entries[prop_name]["networks"].add(network_name)
            ntype = normalize_type(prop_type)
            prop_entries[prop_name]["types"].add(ntype)
            if ntype != "Unknown":
                prop_entries[prop_name]["typed_networks"].add(network_name)

        for result_name, result_type in ot_data.get("result_properties", {}).items():
            if result_name not in result_entries:
                result_entries[result_name] = {"networks": set(), "types": set(), "typed_networks": set()}
            result_entries[result_name]["networks"].add(network_name)
            ntype = normalize_type(result_type)
            result_entries[result_name]["types"].add(ntype)
            if ntype != "Unknown":
                result_entries[result_name]["typed_networks"].add(network_name)

    global_rst_lines.append("Properties")
    global_rst_lines.append('"' * len("Properties"))
    global_rst_lines.append("")
    if prop_entries:
        global_rst_lines.append(".. list-table::")
        global_rst_lines.append("   :header-rows: 1")
        global_rst_lines.append("")
        global_rst_lines.append("   * - Name")
        global_rst_lines.append("     - Value Types")

        rendered_any_property = False
        for prop_name in sorted(prop_entries.keys()):
            networks_seen = sorted(prop_entries[prop_name]["networks"])
            all_types_seen = sorted({normalize_type(t) for t in prop_entries[prop_name]["types"]})
            known_types_seen = filter_unknown_types(all_types_seen)
            types_seen = known_types_seen if known_types_seen else ["Unknown"]

            rendered_any_property = True
            presence_conflict = len(networks_seen) != len(all_network_names)
            type_conflict = len(known_types_seen) > 1
            conflict_flags = []
            if presence_conflict:
                conflict_flags.append("presence")
                property_presence_conflicts.append({
                    "object_type": ot,
                    "property": prop_name,
                    "networks_with_property": networks_seen,
                    "missing_in_networks": [n for n in all_network_names if n not in networks_seen],
                })
            if type_conflict:
                conflict_flags.append("type")
                property_type_conflicts.append({
                    "object_type": ot,
                    "property": prop_name,
                    "types": known_types_seen,
                    "networks_with_property": networks_seen,
                })

            global_rst_lines.append(f"   * - ``{prop_name}``")
            global_rst_lines.append(f"     - ``{' | '.join(types_seen)}``")
        if not rendered_any_property:
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.append("No properties found.")
    else:
        global_rst_lines.append("No properties found.")
    global_rst_lines.append("")

    global_rst_lines.append("Result Properties")
    global_rst_lines.append('"' * len("Result Properties"))
    global_rst_lines.append("")
    if result_entries:
        global_rst_lines.append(".. list-table::")
        global_rst_lines.append("   :header-rows: 1")
        global_rst_lines.append("")
        global_rst_lines.append("   * - Name")

        rendered_any_result = False
        for result_name in sorted(result_entries.keys()):
            networks_seen = sorted(result_entries[result_name]["networks"])
            all_types_seen = sorted({normalize_type(t) for t in result_entries[result_name]["types"]})
            known_types_seen = filter_unknown_types(all_types_seen)

            rendered_any_result = True
            presence_conflict = len(networks_seen) != len(all_network_names)
            type_conflict = len(known_types_seen) > 1
            conflict_flags = []
            if presence_conflict:
                conflict_flags.append("presence")
                result_presence_conflicts.append({
                    "object_type": ot,
                    "result_property": result_name,
                    "networks_with_result_property": networks_seen,
                    "missing_in_networks": [n for n in all_network_names if n not in networks_seen],
                })
            if type_conflict:
                conflict_flags.append("type")
                result_type_conflicts.append({
                    "object_type": ot,
                    "result_property": result_name,
                    "types": known_types_seen,
                    "networks_with_result_property": networks_seen,
                })

            global_rst_lines.append(f"   * - ``{result_name}``")

        if not rendered_any_result:
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.pop()
            global_rst_lines.append("No result properties found.")
    else:
        global_rst_lines.append("No result properties found.")
    global_rst_lines.append("")

global_rst_text = "\n".join(global_rst_lines) + "\n"
global_output_path = output_dir / "object_types_props_results_snippet_global.rst"
global_output_path.write_text(global_rst_text, encoding="utf-8")

conflict_report = {
    "networks": all_network_names,
    "summary": {
        "property_presence_conflicts": len(property_presence_conflicts),
        "property_type_conflicts": len(property_type_conflicts),
        "result_presence_conflicts": len(result_presence_conflicts),
        "result_type_conflicts": len(result_type_conflicts),
    },
    "property_presence_conflicts": property_presence_conflicts,
    "property_type_conflicts": property_type_conflicts,
    "result_presence_conflicts": result_presence_conflicts,
    "result_type_conflicts": result_type_conflicts,
}

conflicts_output_path = output_dir / "object_types_global_conflicts.json"
conflicts_output_path.write_text(json.dumps(conflict_report, indent=2), encoding="utf-8")

global_mapped_text, mapped_count, error_count = map_object_type_headings(
    global_rst_text,
    table_name_mapping,
)
global_mapped_output_path = output_dir / "object_types_props_results_snippet_global_mapped.rst"
global_mapped_output_path.write_text(global_mapped_text, encoding="utf-8")

table_map_output_path = output_dir / "object_type_table_map_independent.json"
table_map_output_path.write_text(
    json.dumps(table_name_mapping, indent=2, sort_keys=True),
    encoding="utf-8",
)

print(f"Global RST: {global_output_path}")
print(f"Global mapped RST: {global_mapped_output_path}")
print(f"Global conflict report: {conflicts_output_path}")
print(f"Independent table map: {table_map_output_path}")
print(f"Property presence conflicts: {len(property_presence_conflicts)}")
print(f"Property type conflicts: {len(property_type_conflicts)}")
print(f"Result presence conflicts: {len(result_presence_conflicts)}")
print(f"Result type conflicts: {len(result_type_conflicts)}")
print(f"Mapped headings (global): {mapped_count}")
print(f"Headings with ERROR_NO_MAPPING__ (global): {error_count}")

all_reports

=== Collecting network: Water ===


[2026-06-26 11:26:50,035] INFO in sir3stoolkit.core.wrapper: New model is created with the model identifier: M-11-0-1
[2026-06-26 11:26:50,039] INFO in sir3stoolkit.core.wrapper: Now you can make changes to the model
[2026-06-26 11:26:50,254] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:26:50,264] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:26:50,264] INFO in sir3stoolkit.core.wrapper: Transaction has ended. Please open up a new transaction if you want to make further changes
[2026-06-26 11:26:56,738] INFO in sir3stoolkit.core.wrapper: Changes saved successfully
[2026-06-26 11:26:56,830] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 5688220107825859679
[2026-06-26 11:26:56,957] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 5058211371704149893
[2026-06-26 11:26:57,027] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 46126

=== Collecting network: DistrictHeating ===


[2026-06-26 11:27:21,334] INFO in sir3stoolkit.core.wrapper: New model is created with the model identifier: M-10-0-1
[2026-06-26 11:27:21,338] INFO in sir3stoolkit.core.wrapper: Now you can make changes to the model
[2026-06-26 11:27:21,346] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:27:21,367] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:27:21,370] INFO in sir3stoolkit.core.wrapper: Transaction has ended. Please open up a new transaction if you want to make further changes
[2026-06-26 11:27:25,171] INFO in sir3stoolkit.core.wrapper: Changes saved successfully
[2026-06-26 11:27:25,191] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 4855968828564406472
[2026-06-26 11:27:25,197] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 5545359002192974015
[2026-06-26 11:27:25,202] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 55148

=== Collecting network: Gas ===


[2026-06-26 11:27:39,348] INFO in sir3stoolkit.core.wrapper: New model is created with the model identifier: M-10-0-1
[2026-06-26 11:27:39,351] INFO in sir3stoolkit.core.wrapper: Now you can make changes to the model
[2026-06-26 11:27:39,361] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:27:39,378] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:27:39,381] INFO in sir3stoolkit.core.wrapper: Transaction has ended. Please open up a new transaction if you want to make further changes
[2026-06-26 11:27:42,669] INFO in sir3stoolkit.core.wrapper: Changes saved successfully
[2026-06-26 11:27:42,683] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 5628281028760566039
[2026-06-26 11:27:42,689] ERROR in sir3stoolkit.core.wrapper: Error : AirVessel: Element Type cannot be inserted into this Network Type
[2026-06-26 11:27:42,693] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 5288

=== Collecting network: Steam ===


[2026-06-26 11:27:52,379] INFO in sir3stoolkit.core.wrapper: New model is created with the model identifier: M-10-0-1
[2026-06-26 11:27:52,380] INFO in sir3stoolkit.core.wrapper: Now you can make changes to the model
[2026-06-26 11:27:52,391] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:27:52,403] INFO in sir3stoolkit.core.wrapper: New node added
[2026-06-26 11:27:52,406] INFO in sir3stoolkit.core.wrapper: Transaction has ended. Please open up a new transaction if you want to make further changes
[2026-06-26 11:27:55,264] INFO in sir3stoolkit.core.wrapper: Changes saved successfully
[2026-06-26 11:27:55,277] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 4721342691812129628
[2026-06-26 11:27:55,281] ERROR in sir3stoolkit.core.wrapper: Error : AirVessel: Element Type cannot be inserted into this Network Type
[2026-06-26 11:27:55,287] INFO in sir3stoolkit.core.wrapper: Element inserted successfully into the model with Tk: 4663

Global RST: C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_types_props_results_snippet_global.rst
Global mapped RST: C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_types_props_results_snippet_global_mapped.rst
Global conflict report: C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_types_global_conflicts.json
Independent table map: C:\Users\aUsername\3S\sir3stoolkit\docs\source\tutorials\SIR3S_Model_Mantle\TutorialTest_Assets\object_types\output\object_type_table_map_independent.json
Property presence conflicts: 0
Property type conflicts: 0
Result presence conflicts: 0
Result type conflicts: 0
Mapped headings (global): 146
Headings with ERROR_NO_MAPPING__ (global): 3


[{'network': 'Water',
  'network_output': 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\docs\\source\\tutorials\\SIR3S_Model_Mantle\\TutorialTest_Assets\\object_types\\output\\object_types_props_results_snippet_Water.rst',
  'network_mapped_output': 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\docs\\source\\tutorials\\SIR3S_Model_Mantle\\TutorialTest_Assets\\object_types\\output\\object_types_props_results_snippet_Water_mapped.rst',
  'object_types': 149,
  'mapped_headings': 146,
  'mapping_errors': 3},
 {'network': 'DistrictHeating',
  'network_output': 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\docs\\source\\tutorials\\SIR3S_Model_Mantle\\TutorialTest_Assets\\object_types\\output\\object_types_props_results_snippet_DistrictHeating.rst',
  'network_mapped_output': 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\docs\\source\\tutorials\\SIR3S_Model_Mantle\\TutorialTest_Assets\\object_types\\output\\object_types_props_results_snippet_DistrictHeating_mapped.rst',
  'object_types': 149,
  'mapped_headings